# Gemma exception-contract audit: proposed competition update
Jacek Adamek - 16 October 2026. No Writeup is changed or submitted.
Run from this repository root or dated update directory. Install requirements.txt.
The notebook verifies frozen evidence and prints the supported results.


In [ ]:
import os, json, sys, subprocess
from pathlib import Path
REPO = Path.cwd().resolve()
if not (REPO/'releases/2026-10-16').is_dir(): REPO = REPO.parent
RELEASE = Path(os.environ.get('GEMMA_RELEASE', str(REPO/'releases/2026-10-16'))).resolve()
assert (RELEASE/'verify_release.py').is_file(), 'Configure GEMMA_RELEASE'
subprocess.run([sys.executable, str(RELEASE/'verify_release.py')], check=True, capture_output=True)
resource = RELEASE/'outputs/gemma-importguard/results/exception-contract-resource'
calibration = json.loads((resource/'calibration-evaluation.json').read_text())
model = json.loads((resource/'model-evaluation.json').read_text())
posthoc = json.loads((resource/'posthoc-v2-evaluation.json').read_text())
assert calibration['weak_pass_count'] == 42 and calibration['contract_pass_count'] == 7
assert model['weak_pass_count'] == model['contract_pass_count'] == 6
assert posthoc['model_passes'] == 5 and posthoc['mutants_rejected'] == 36
print('Original ECA: 36/36 faults pass weak gate; v1 rejects 35/36; post-hoc v2 rejects 36/36.')
print('Model outputs: 6/6 initial checks, 5/6 post-hoc v2. Original ItsDangerous failure remains.')


In [ ]:
tenacity = json.loads((RELEASE/'outputs/gemma-feedback-v4/evaluated.json').read_text())
packaging = json.loads((RELEASE/'outputs/gemma-branch-feedback-v5/evaluated.json').read_text())
for label, item, expected in [('Tenacity', tenacity[0], [142,8,9]), ('Packaging', packaging, [74,7,9])]:
    counts = []
    for group in ['upstream','examples','withheld']:
        evidence = item['runs'][group]['evidence']
        assert evidence['complete'] and evidence['accepted']
        counts.append(sum(len(v) for v in evidence['groups'].values()))
    assert counts == expected
    print(label, 'upstream/examples/withheld:', counts, 'total:', sum(counts))
diff = json.loads((RELEASE/'outputs/gemma-branch-feedback-v5/differential-result.json').read_text())
assert diff['tested_names'] == 1245 and diff['all_agree'] and diff['post_hoc']
print('Post-hoc Packaging comparison:', diff['tested_names'], 'names; differences:', len(diff['differences']))
print('Read the records and failed earlier attempts; these are final guided patches, not first attempts.')


In [ ]:
plan = subprocess.run([sys.executable, str(RELEASE/'generate_repairs.py'), '--plan'], check=True, capture_output=True, text=True)
record = json.loads(plan.stdout)
assert record['seed'] == 20261016 and record['context'] == 4096
print('Final assisted prompt hashes:', {x['id']:x['prompt_sha256'] for x in record['tasks']})
print('Fresh inference is optional and requires separately acquired weights/runtime; no new generation run here.')


## Full independent replay
From the release directory, run reproduce.py into a new directory for all 102 ECA variants.
Use replay_repairs.py with the four hash-pinned upstream archives from FETCH_UPSTREAM.md
to rerun fixed, parent and reviewed model versions. A fresh-copy audit already repeated
both paths in isolated Python 3.12 environments; results remain bounded by their tests.
The submitted paper is unchanged; writeup-update.txt is the proposed replacement text.
